# freMTPL2 Scaling Laws for Actuaries

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RonRichman/frmtpl-scaling-laws/blob/main/notebooks/01_colab_frmtpl_scaling.ipynb)

This notebook runs the open-source companion experiment for the paper [Scaling Laws, Tabular Data and Actuarial Ratemaking Models](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=6073948).

The workflow uses Mario Wüthrich's corrected public freMTPL2 frequency data, the Wüthrich-Merz Listing 5.2 learning/test split, an exposure-scaled Poisson frequency objective, nested training fractions, three seeds, and compact Keras versions of the model families from the paper. For dataset definitions and cleaning details, see Appendix B of Wüthrich and Merz: https://link.springer.com/book/10.1007/978-3-031-12409-9.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/RonRichman/frmtpl-scaling-laws.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/frmtpl-scaling-laws")


def find_package_dir(start):
    start = Path(start).resolve()
    candidates = [start, start.parent]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "frmtpl_scaling").exists():
            return candidate
    return None


package_dir = find_package_dir(Path.cwd())
if package_dir is None:
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    package_dir = REPO_DIR

os.chdir(package_dir)
print(f"Working directory: {Path.cwd()}")

!pip install -q -e ".[dev]"

## Data Check

If the materialized CSV is not present in the clone, regenerate it from Mario Wüthrich's corrected RDA file.

In [ ]:
from pathlib import Path

if not Path('data/FRMTPL.csv').exists():
    !Rscript scripts/prepare_wuthrich_data.R

## Smoke Run

Use this first to verify the environment. It trains only GLM and FFN for two small fractions, one seed, and one epoch. TensorFlow may print CUDA-driver messages on a CPU-only runtime; those messages are informational if training continues.

In [ ]:
!python scripts/run_experiment.py --smoke

## Full Public Sweep

The default run uses six nested fractions and three seeds for each compact model family. On a free Colab GPU this can take a while; reduce `--epochs` if needed.

In [ ]:
!python scripts/run_experiment.py --epochs 30

In [ ]:
!python scripts/make_figures.py

In [ ]:
import pandas as pd

ensemble = pd.read_csv('results/ensemble_scores.csv')
fits = pd.read_csv('results/scaling_fits.csv')
display(ensemble.head())
display(fits)

Interpretation: `alpha` is the fitted data-scaling exponent in `L(N) = L_inf + A N^-alpha`. Within this public experiment, larger `alpha` means a model family improves faster as more freMTPL2 training rows are added.